# Meqsed

### meqsed odurki bizde studyinstanceid var ve onlarin ozlerine aid feature var. Biz sample submission fayli ucun herbir studyinstanceid ucun 12 dene feature cixarmaliyiq ve vermeliyik. Yalniz test faylinin icinde knee MRI sekilleri var

### .dcm - MRI sekli; train.csv - hansi xesteliklerin oldugu, hansi studylerin icinde hansi MRI seriesler var ve bu ne tip goruntudur

In [ ]:
"""import os

csv_files = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.lower().endswith('.csv'):
            csv_files.append(os.path.join(dirname, filename))

for file in csv_files:
    print(file)"""

In [ ]:
import os

path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"
sumitems = 0 

for item in os.listdir(path):
    sumitems += 1
print(sumitems)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.model_download("qwen-lm/qwen2.5/transformers/7b")

print("Path to model files:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
from transformers import TrainingArguments, Trainer

# Data analysis

In [ ]:
new_train_file = pd.read_csv("/kaggle/input/datasets/thesharifzade/newtrainfile/new_train_checkpoint.csv")

In [ ]:
train_series = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv")

In [ ]:
train_series.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
from transformers import TrainingArguments, Trainer

In [ ]:
!pip uninstall -y torchao

In [ ]:
!pip install --no-index "/kaggle/input/datasets/thesharifzade/torchao/torchao_pkg/torchao-0.16.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl"

# Training Dataset

In [ ]:
train_dataset = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")

In [ ]:
train_dataset.head()

In [ ]:
train_dataset.nunique()

In [ ]:
train_dataset.shape

In [ ]:
train_dataset.dtypes

In [ ]:
train_dataset.info()

In [ ]:
train_dataset.isnull().sum()

In [ ]:
sns.heatmap(train_dataset.isnull(), cbar=False)
plt.show()

In [ ]:
train_dataset.describe()

In [ ]:
train_dataset.describe(include='all')

In [ ]:
train_dataset.corr(numeric_only = True)

In [ ]:
# Correlation matrix
corr = train_dataset.corr(numeric_only=True)

# Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Distribution 
numeric_cols = train_dataset.select_dtypes(include="number").columns
for i in numeric_cols:
    sns.histplot(train_dataset[i], kde=True)
    plt.show()

In [ ]:
for i in numeric_cols:
    sns.boxplot(x=train_dataset[i])
    plt.show()

# Training dataset series

In [ ]:
train_dataset_series = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv")

In [ ]:
train_dataset_series.head()

In [ ]:
train_dataset_series.info()

In [ ]:
train_dataset_series.describe()

In [ ]:
train_dataset_series.describe(include ='all')

In [ ]:
train_dataset_series.shape

In [ ]:
train_dataset_series.dtypes

# Train series (images)

In [ ]:
# burda demeli problem nedir seriesinstance uidler demeli studyinstance uidlerin icindedir
# ona gore birinci studyinstance id aldig sora seriesinstance id
import os

path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

study = os.listdir(path)[0]

print("Study:", study)
print(os.listdir(os.path.join(path, study))[:20])

In [ ]:
series = os.listdir(os.path.join(path, study))[0]

series_path = os.path.join(path, study, series)

print("Series:", series)
print(os.listdir(series_path)[:10])

In [ ]:
import os
import pydicom
import matplotlib.pyplot as plt

path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

study = os.listdir(path)[0]
series = os.listdir(os.path.join(path, study))[0]

series_path = os.path.join(path, study, series)

dcm_file = [f for f in os.listdir(series_path) if f.endswith(".dcm")][0]

dcm = pydicom.dcmread(os.path.join(series_path, dcm_file))

plt.imshow(dcm.pixel_array, cmap="gray")
plt.axis("off")
plt.show()

# Training images with DinoV2

In [ ]:
from transformers import AutoImageProcessor, AutoModel

model_path = "/kaggle/input/models/metaresearch/dinov2/pytorch/large/1"

# processor burda sekili dinoV2-nin basa duseceyi formata cevirir
# 512x512 -> RGB -> resize / normalize -> tensor
processor = AutoImageProcessor.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path).cuda()
model.eval()

In [ ]:
import os
import pydicom
import numpy as np
from PIL import Image
import torch

path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

# bir Study
study = os.listdir(path)[0]

# onun bir Series-i
series = os.listdir(os.path.join(path, study))[0]

series_path = os.path.join(path, study, series)

# bir DICOM
dcm_file = [f for f in os.listdir(series_path) if f.endswith(".dcm")][0]

dcm = pydicom.dcmread(os.path.join(series_path, dcm_file))

# DICOM -> image
img = dcm.pixel_array.astype(np.float32)

# normalize 0-255
img = (img - img.min()) / (img.max() - img.min()) * 255
img = img.astype(np.uint8)

# grayscale -> RGB
img = Image.fromarray(img).convert("RGB")

In [ ]:
inputs = processor(
    images=img,
    return_tensors="pt"
).to("cuda")

print(inputs["pixel_values"].shape)

In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

print(outputs.last_hidden_state.shape)

In [ ]:
feature = outputs.last_hidden_state[:, 0, :]

print(feature.shape)

# Bidene experiment train_series ile train_series.csv arasindaki baglantini qurmaq ucun

In [ ]:
"""
Demeli seriesler var ve onlarin icinde sekiller var
Biz indi birinci 1-ci seriesi goturub, icindeki sekilleri analiz edib, feature engineering edirik
ordaki fluit sensitive zad onlari labelleyeik ki 1024 dene yox cunki biz 1024x1024 elemisik
1027 yeni o ucunude bizim ucun versin dinoV2 yoxsa bele ozu bele dusune bilmir
Belke sonra experiment ucun 1024x1024 de qalib, ondan sonra qwen ile analiz edib basqa curde elemek olar

"""

In [ ]:
import pandas as pd
import os

df = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"
)

train_path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

row = df.iloc[0]

study_id = row["StudyInstanceUID"]
series_id = row["SeriesInstanceUID"]

series_path = os.path.join(train_path, study_id, series_id)

print("Series:", series_id)
print("Folder exists:", os.path.exists(series_path))
print("DICOM count:", len([f for f in os.listdir(series_path) if f.endswith(".dcm")]))

In [ ]:
# burda ise biz processing edirik bir novu
features = []

dcm_files = sorted([
    f for f in os.listdir(series_path)
    if f.endswith(".dcm")
])

for file in dcm_files:
    dcm = pydicom.dcmread(os.path.join(series_path, file))

    img = dcm.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min()) * 255
    img = img.astype(np.uint8)

    img = Image.fromarray(img).convert("RGB")

    inputs = processor(images=img, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model(**inputs)

    feature = output.last_hidden_state[:, 0, :]
    features.append(feature.cpu())

features = torch.cat(features, dim=0)

print(features.shape)

In [ ]:
series_feature = features.mean(dim=0)

print(series_feature.shape)

In [ ]:
# Burda just bidene row-un fluit sensitive zad o melumatlarini bura yazir
row = df[df["SeriesInstanceUID"] == series_id].iloc[0]

fluid = row["Fluid_Sensitive"]
fat = row["Fat_Suppression"]
plane = row["Anatomical_Plane"]

print(fluid, fat, plane)

In [ ]:
# Burda uje o 3 dene featurede elave olundu bizim ucun 
plane_map = {
    "Sagittal": 0,
    "Coronal": 1,
    "Axial": 2
}

plane_value = plane_map[plane]

metadata = torch.tensor([
    fluid,
    fat,
    plane_value
], dtype=torch.float32)

final_feature = torch.cat([
    series_feature,
    metadata
])

print(final_feature.shape)

In [ ]:
# Nece dene series oldugunu gosterir bu
study_rows = df[df["StudyInstanceUID"] == study_id]

print(len(study_rows))
print(study_rows["SeriesInstanceUID"].tolist())

In [ ]:
for series_id in study_rows["SeriesInstanceUID"]:
    row = df[df["SeriesInstanceUID"] == series_id].iloc[0]

    print(
        "| Series ID:",series_id,
        "| Fluid:", row["Fluid_Sensitive"],
        "| Fat:", row["Fat_Suppression"],
        "| Plane:", row["Anatomical_Plane"]
    )

In [ ]:
series_features = {}

for series_id in study_rows["SeriesInstanceUID"]:

    series_path = os.path.join(
        train_path,
        study_id,
        series_id
    )

    features = []

    dcm_files = sorted([
        f for f in os.listdir(series_path)
        if f.endswith(".dcm")
    ])

    for file in dcm_files:
        dcm = pydicom.dcmread(
            os.path.join(series_path, file)
        )

        img = dcm.pixel_array.astype(np.float32)

        # normalize
        img = (img - img.min()) / (img.max() - img.min()) * 255
        img = img.astype(np.uint8)

        img = Image.fromarray(img).convert("RGB")

        inputs = processor(
            images=img,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            output = model(**inputs)

        feature = output.last_hidden_state[:, 0, :]
        features.append(feature.cpu())

    # DICOM-ları birləşdir
    features = torch.cat(features, dim=0)

    # Series feature
    series_feature = features.mean(dim=0)

    series_features[series_id] = series_feature

    print(series_id, "→", series_feature.shape)

In [ ]:
plane_map = {
    "Sagittal": 0,
    "Coronal": 1,
    "Axial": 2
}

final_series_features = {}

for series_id, feature in series_features.items():

    row = df[df["SeriesInstanceUID"] == series_id].iloc[0]

    metadata = torch.tensor([
        row["Fluid_Sensitive"],
        row["Fat_Suppression"],
        plane_map[row["Anatomical_Plane"]]
    ], dtype=torch.float32)

    final_feature = torch.cat([feature, metadata])

    final_series_features[series_id] = final_feature

    print(series_id, final_feature.shape)

In [ ]:
study_feature = torch.stack(
    list(final_series_features.values())
).mean(dim=0)

print(study_feature.shape)

In [ ]:
label_cols = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

train_df = train_dataset

train_labeled = train_df.dropna(subset=label_cols)

print(len(train_df))
print(len(train_labeled))

In [ ]:
print(train_labeled[label_cols].head())

In [ ]:
label_cols = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

print("Total studies:", len(train_df))
print("Labeled studies:", train_df[label_cols].notna().all(axis=1).sum())

In [ ]:
labeled_studies = train_df[
    train_df[label_cols].notna().all(axis=1)
]

study_id = labeled_studies.iloc[0]["StudyInstanceUID"]

print("Study:", study_id)

In [ ]:
study_series = df[
    df["StudyInstanceUID"] == study_id
]

print("Series count:", len(study_series))
print(study_series[
    ["SeriesInstanceUID", "Fluid_Sensitive",
     "Fat_Suppression", "Anatomical_Plane"]
])

In [ ]:
study_series_features = []

for _, row in study_series.iterrows():

    series_id = row["SeriesInstanceUID"]

    series_path = os.path.join(
        train_path,
        study_id,
        series_id
    )

    features = []

    dcm_files = sorted([
        f for f in os.listdir(series_path)
        if f.endswith(".dcm")
    ])

    for file in dcm_files:

        dcm = pydicom.dcmread(
            os.path.join(series_path, file)
        )

        img = dcm.pixel_array.astype(np.float32)

        img = (
            (img - img.min()) /
            (img.max() - img.min() + 1e-8)
            * 255
        ).astype(np.uint8)

        img = Image.fromarray(img).convert("RGB")

        inputs = processor(
            images=img,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            output = model(**inputs)

        feature = output.last_hidden_state[:, 0, :]
        features.append(feature.cpu())

    # DICOM-ları → Series feature
    features = torch.cat(features, dim=0)
    series_feature = features.mean(dim=0)

    study_series_features.append(series_feature)

    print(
        series_id,
        "DICOMs:", len(dcm_files),
        "Feature:", series_feature.shape
    )

In [ ]:
study_feature = torch.stack(
    study_series_features
).mean(dim=0)

print("Study feature:", study_feature.shape)

In [ ]:
labels = labeled_studies[
    labeled_studies["StudyInstanceUID"] == study_id
][label_cols].iloc[0].values.astype("float32")

labels = torch.tensor(labels)

print("Feature:", study_feature.shape)
print("Labels:", labels.shape)
print(labels)

In [ ]:
X = []
y = []

for _, study_row in labeled_studies.iterrows():

    study_id = study_row["StudyInstanceUID"]

    study_series = df[
        df["StudyInstanceUID"] == study_id
    ]

    series_features = []

    for _, series_row in study_series.iterrows():

        series_id = series_row["SeriesInstanceUID"]

        series_path = os.path.join(
            train_path,
            study_id,
            series_id
        )

        features = []

        dcm_files = sorted([
            f for f in os.listdir(series_path)
            if f.endswith(".dcm")
        ])

        for file in dcm_files:

            dcm = pydicom.dcmread(
                os.path.join(series_path, file)
            )

            img = dcm.pixel_array.astype(np.float32)

            img = (
                (img - img.min()) /
                (img.max() - img.min() + 1e-8)
                * 255
            ).astype(np.uint8)

            img = Image.fromarray(img).convert("RGB")

            inputs = processor(
                images=img,
                return_tensors="pt" # Pytorch tensor format
            ).to("cuda")

            with torch.no_grad():
                output = model(**inputs)

            feature = output.last_hidden_state[:, 0, :]
            features.append(feature.cpu())

        # DICOM → Series
        features = torch.cat(features, dim=0)
        series_feature = features.mean(dim=0)

        series_features.append(series_feature)

    # Series → Study
    study_feature = torch.stack(series_features).mean(dim=0)

    # Label
    labels = study_row[label_cols].values.astype("float32")

    X.append(study_feature)
    y.append(torch.tensor(labels))

    print("Done:", study_id)

In [ ]:
X = torch.stack(X)
y = torch.stack(y)

print("X:", X.shape)
print("y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)

# Knee Classifier

In [ ]:
import torch.nn as nn

class KneeClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 12)
        )

    def forward(self, x):
        return self.network(x)

KneeClassifier = KneeClassifier().cuda()

print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    KneeClassifier.parameters(),
    lr=1e-4
)

In [ ]:
X_train = X_train.cuda()
y_train = y_train.cuda()

X_val = X_val.cuda()
y_val = y_val.cuda()

epochs = 200

for epoch in range(epochs):

    KneeClassifier.train()

    optimizer.zero_grad()

    outputs = KneeClassifier(X_train)

    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    """print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {loss.item():.4f}"
    )"""

In [ ]:
# 12,12 verdi yeni 12 valuation datasi , 12 features uje var
KneeClassifier.eval()

with torch.no_grad():
    val_logits = KneeClassifier(X_val)

    val_probs = torch.sigmoid(val_logits)

print(val_probs.shape)

In [ ]:
predictions = (val_probs >= 0.5).float()

print(predictions)
print(y_val)

In [ ]:
# Evaluate elemek ucundur! 
# neticeler zeif oldu, cunki hal hazirda bizim elimizde cemi 58 data var idi, amma olsun indi qwen ile datalari coxaldib
# yeniden experiment elemek olar
from sklearn.metrics import f1_score, roc_auc_score

y_true = y_val.cpu().numpy()
y_pred = predictions.cpu().numpy()
y_prob = val_probs.cpu().numpy()

for i, label in enumerate(label_cols):

    f1 = f1_score(
        y_true[:, i],
        y_pred[:, i],
        zero_division=0
    )

    print(f"{label}: F1 = {f1:.4f}")

In [ ]:
for i, label in enumerate(label_cols):

    try:
        auc = roc_auc_score(
            y_true[:, i],
            y_prob[:, i]
        )

        print(f"{label}: AUC = {auc:.4f}")

    except ValueError:
        print(f"{label}: AUC hesablamaq mümkün deyil")

# Test Series (images)

In [ ]:
import os

folder = "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series/"

dcm_files_test = []

for root, dirs, files in os.walk(folder):
    for file in files:
        if file.endswith(".dcm"):
            dcm_files_test.append(os.path.join(root, file))

print(len(dcm_files_test))

In [ ]:
# Demeli sekillerin uzerinde olan idler- Series instance UID uzerinden aparir
import pydicom
import matplotlib.pyplot as plt

for file in dcm_files_test[:5]:
    dcm = pydicom.dcmread(file)
    image = dcm.pixel_array

    plt.figure(figsize=(5, 5))
    plt.imshow(image, cmap="gray")
    plt.title(file)
    plt.axis("off")
    plt.show()

# Test Dataset (Text)

In [ ]:
test_dataset = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv")

In [ ]:
len(test_dataset)

In [ ]:
test_dataset.info()

In [ ]:
train_ids = set(train_dataset["StudyInstanceUID"])
test_ids = set(test_dataset["StudyInstanceUID"])

overlap = train_ids & test_ids

print("Train IDs:", len(train_ids))
print("Test IDs:", len(test_ids))
print("Overlap:", len(overlap))

if overlap:
    print("Eyni ID-lər tapıldı:")
    print(list(overlap)[:10])

In [ ]:
test_dataset.shape

# Submission File

In [ ]:
submission_file = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/sample_submission.csv")

In [ ]:
submission_file.head()

In [ ]:
len(submission_file)

In [ ]:
submission_file.columns

In [ ]:
submission_file['StudyInstanceUID']

## Creating my own tokenizer

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/thesharifzade/pharmaconerdataset"):
    print(root)
    print(files[:10])

In [ ]:
# so we use the sentencepiece
# download the dataset
from datasets import load_from_disk

cantemistnerdataset = load_from_disk(
    "/kaggle/input/datasets/thesharifzade/cantemistdatasetfile/cantemist-ner"
)

print(cantemistnerdataset)

In [ ]:
# upload pharmaconerdataset
from datasets import load_from_disk
pharmaconerdataset = load_from_disk("/kaggle/input/datasets/thesharifzade/pharmaconerdataset/pharmaco-ner")
print(pharmaconerdataset)

In [ ]:
from datasets import concatenate_datasets

train1 = cantemistnerdataset["train"].remove_columns(["ner_tags"])
train2 = pharmaconerdataset["train"].remove_columns(["ner_tags"])

train_ds = concatenate_datasets([
    train1,
    train2
])

print(train_ds)

In [ ]:
# train_ds artıq hazırdır

texts = [
    " ".join(example["tokens"])
    for example in train_ds
]

with open("/kaggle/working/cantemist_train.txt", "w", encoding="utf-8") as f:
    for text in texts:
        f.write(text + "\n")

print("Samples:", len(texts))
print(texts[0])

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="/kaggle/working/cantemist_train.txt",
    model_prefix="/kaggle/working/cantemist_tokenizer",
    model_type="bpe",
    vocab_size=8000,
    character_coverage=0.99995,

    normalization_rule_name="identity",
    add_dummy_prefix=True,
    remove_extra_whitespaces=False,

    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_by_number=True,
    split_digits=True,

    max_sentencepiece_length=16,
    allow_whitespace_only_pieces=True,
    byte_fallback=True,

    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3,

    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>",
    pad_piece="<pad>",

    hard_vocab_limit=True,
    input_sentence_size=0,
    shuffle_input_sentence=True,
)

In [ ]:
sp = spm.SentencePieceProcessor(
    model_file="/kaggle/working/cantemist_tokenizer.model"
)

text = texts[0]

print(sp.encode(text, out_type=str))
print(sp.encode(text, out_type=int))

In [ ]:
"""model = AutoModelForCausalLM.from_pretrained(
    ai_model,
    torch_dtype=torch.float16
)"""

In [ ]:
"""qwen_messages = []

for item in qwen_data:
    qwen_messages.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "Read the medical report and predict the 12 labels. "
                    "For each label, output only 0 or 1."
                )
            },
            {
                "role": "user",
                "content": item["input"]
            },
            {
                "role": "assistant",
                "content": str(item["output"])
            }
        ]
    })

print(qwen_messages[0])"""

In [ ]:
"""from datasets import Dataset

dataset = Dataset.from_list(qwen_messages)

print(dataset)"""

In [ ]:
"""def tokenize(example):
    text = ""

    for message in example["messages"]:
        text += message["content"] + "\n"

    input_ids = sp.encode(text, out_type=int)

    input_ids = input_ids[:1024]

    padding_length = 1024 - len(input_ids)

    input_ids += [0] * padding_length

    attention_mask = [1] * (1024 - padding_length) + [0] * padding_length

    labels = input_ids.copy()

    labels = [
        token if mask == 1 else -100
        for token, mask in zip(labels, attention_mask)
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)"""

In [ ]:
"""print(tokenized_dataset[0]["input_ids"][:5])"""

In [ ]:
"""# input ids -> result of tokenization(with ids)
# attention_mask -> which padding, which real token
# label -> the target
print(tokenized_dataset.column_names)
print(tokenized_dataset[0].keys())"""

In [ ]:
"""print("input_ids:", len(tokenized_dataset[0]["input_ids"]))
print("attention_mask:", len(tokenized_dataset[0]["attention_mask"]))
print("labels:", len(tokenized_dataset[0]["labels"]))"""

In [ ]:
"""# PEFT method : free entire LLM, add tiny layers
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
"""

In [ ]:
"""training_args = TrainingArguments(
    output_dir="./qwen_knee",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    logging_steps=1,
    logging_strategy = "steps",
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)"""

In [ ]:
"""trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)"""

In [ ]:
#trainer.train()

In [ ]:
"""import matplotlib.pyplot as plt

logs = trainer.state.log_history

steps = []
losses = []

for log in logs:
    if "loss" in log:
        steps.append(log["step"])
        losses.append(log["loss"])

plt.plot(steps, losses)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid()
plt.show()"""

In [ ]:
#print(trainer.state.log_history)

In [ ]:
#trainer.save_model("./finetuningazesavedversion")

# Check the fine tuned version (is working yes or no?)

In [ ]:
#file_path = "/kaggle/input/datasets/thesharifzade/finetuningazerbaijanversionrsnaknee/finetuningazesavedversion"

In [ ]:
"""import os

print(os.listdir(file_path))"""

In [ ]:
"""from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b/1"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype="auto",
    device_map="auto"
)"""

In [ ]:
"""
model = PeftModel.from_pretrained(
    model,
    file_path
)"""

In [ ]:
"""prompt = "Your test prompt here"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))"""

# Check the fine tuned version for train labels with actual data and prediction 

In [ ]:
#train_dataset = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")

In [ ]:
"""# yalnız 12 label-i olan 58 sətri götürəcəyik
labeled_data = train_dataset.dropna()

print(labeled_data.shape)"""

In [ ]:
"""labels = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

qwen_data = []

for _, row in labeled_data.iterrows():
    qwen_data.append({
        "input": row["Report"],
        "output": {label: int(row[label]) for label in labels}
    })

print(qwen_data[0])"""

In [ ]:
"""qwen_messages = []

for item in qwen_data:
    qwen_messages.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "Read the medical report and predict the 12 labels. "
                    "For each label, output only 0 or 1."
                )
            },
            {
                "role": "user",
                "content": item["input"]
            },
            {
                "role": "assistant",
                "content": str(item["output"])
            }
        ]
    })

print(qwen_messages[0])"""

In [ ]:
"""from datasets import Dataset, concatenate_datasets

# 1. Qwen dataset
dataset = Dataset.from_list(qwen_messages)

print(dataset)"""

In [ ]:
"""import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="/kaggle/working/cantemist_train.txt",
    model_prefix="/kaggle/working/cantemist_tokenizer",
    model_type="bpe",
    vocab_size=8000,
    character_coverage=0.99995,

    normalization_rule_name="identity",
    add_dummy_prefix=True,
    remove_extra_whitespaces=False,

    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_by_number=True,
    split_digits=True,

    max_sentencepiece_length=16,
    allow_whitespace_only_pieces=True,
    byte_fallback=True,

    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3,

    unk_piece="<unk>",
    bos_piece="<s>",
    eos_piece="</s>",
    pad_piece="<pad>",

    hard_vocab_limit=True,
    input_sentence_size=0,
    shuffle_input_sentence=True,
)"""

In [ ]:
"""sp = spm.SentencePieceProcessor(
    model_file="/kaggle/working/cantemist_tokenizer.model"
)

text = texts[0]

print(sp.encode(text, out_type=str))
print(sp.encode(text, out_type=int))"""

In [ ]:
"""def tokenize(example):
    text = ""

    for message in example["messages"]:
        text += message["content"] + "\n"

    input_ids = sp.encode(text, out_type=int)

    input_ids = input_ids[:1024]

    padding_length = 1024 - len(input_ids)

    input_ids += [0] * padding_length

    attention_mask = [1] * (1024 - padding_length) + [0] * padding_length

    labels = input_ids.copy()

    labels = [
        token if mask == 1 else -100
        for token, mask in zip(labels, attention_mask)
    ]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)"""

In [ ]:
"""# input ids -> result of tokenization(with ids)
# attention_mask -> which padding, which real token
# label -> the target
print(tokenized_dataset.column_names)
print(tokenized_dataset[0].keys())"""

In [ ]:
#print(tokenized_dataset[0]["input_ids"][:5])

In [ ]:
#print(tokenized_dataset[0]["attention_mask"][:5])
#print(tokenized_dataset[0]["labels"][:5])

In [ ]:
# Test report terefde problem var labeller oyrenmeyib ona gore bele raw text cixarib 
"""
'Antecedentes Clínicos:\nEsguince rodilla. [DATE].\nHallazgos:\nNo hay alteraciones de señal significativas de la médula ósea.\nLigamentos cruzados y colaterales dentro de límites normales.\nAmputación marginal del cuerpo del menisco lateral. Menisco medial de morfología y señal\nconservada, sin signos de rotura.\nCartílagos de los compartimentos femorotibiales sin alteraciones.\nFina úlcera condral focal de espesor total del aspecto inferior de la vertiente medial de la tróclea\nfemoral con mínimos cambios óseos secundarios. Fenómenos condrales reparativos de la región\ncentral de la simple femoral. Cartílago rotuliano sin alteraciones.\nLeve derrame articular. No hay quistes poplíteos patológicos.\nAumento de señal de la inserción distal del tendón cuadricipital y de la inserción proximal del\ntendón rotuliano, sin signos de rotura.\nNo hay alteraciones de señal de la grasa de Hoffa.\nImpresión:\nAmputación marginal del cuerpo del menisco lateral.\nCondropatía focal grado 4 del aspecto inferior de la vertiente medial de la tróclea femoral .\nFenómenos condrales reparativos de la región central de la tróclea femoral.\nLeve derrame articular.\nLeve tendinosis cuadricipital distal y rotuliana proximal.

"""

In [ ]:
#qwen_data[0]["output"]

In [ ]:
#test_report = qwen_data[0]["input"]

In [ ]:
#test_report

# Analysis more bigger version of qwen

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
from transformers import TrainingArguments, Trainer

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.model_download("qwen-lm/qwen2.5/transformers/7b")

print("Path to model files:", path)

In [ ]:
# quantized version edirik yeni bir novu modelin icindeki parameterler terefden evvel 16 bit tam formatda verirdise
# indi 4 bit formatinda verecek, hesablamalar terefden ozunu zeif apara biler, amma dusunme terefden guclu aparacagina inaniriq



In [ ]:
#!ls -lh /kaggle/input/datasets/thesharifzade/bitsandbytesrsnaazerbaijan

In [ ]:
#!ls -lh /kaggle/input/datasets/thesharifzade/bitsandbytesrsnaazerbaijan/wheels

In [ ]:
#!pip uninstall -y bitsandbytes

In [ ]:
!pip install --no-index --no-deps \
    --find-links=/kaggle/input/datasets/thesharifzade/bitsandbytesrsnaazerbaijan/wheels \
    bitsandbytes==0.50.2

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import seaborn as sns
from transformers import TrainingArguments, Trainer

In [ ]:
!pip install --no-index "/kaggle/input/datasets/thesharifzade/torchao/torchao_pkg/torchao-0.16.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = path

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name) # tokenizer from qwen 14b

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

In [ ]:
tokenizer

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/thesharifzade/pharmaconerdataset"):
    print(root)
    print(files[:10])

In [ ]:
train_dataset = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv")

In [ ]:
# yalnız 12 label-i olan 58 sətri götürəcəyik
labeled_data = train_dataset.dropna()

print(labeled_data.shape)

In [ ]:
train_dataset[:5]

In [ ]:
labels = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

qwen_data = []

for _, row in labeled_data.iterrows():
    qwen_data.append({
        "input": row["Report"],
        "output": {label: int(row[label]) for label in labels}
    })

print(qwen_data[0])

In [ ]:
import json

example = qwen_data[0]

messages = [
    {
        "role": "system",
        "content": """
Read a medical knee MRI report and output exactly 12 labels as JSON.
Use only 0 or 1.

The labels must be exactly:
ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA,
Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.
"""
    },
    {
        "role": "user",
        "content": example["input"]
    },
    {
        "role": "assistant",
        "content": json.dumps(example["output"])
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(text)

In [ ]:
tokens = tokenizer(
    text,
    truncation=True,
    max_length=1024,
    padding=False
)

print(tokens.keys())
print(tokens["input_ids"][:20])
print("Number of tokens:", len(tokens["input_ids"]))

In [ ]:
decoded = tokenizer.decode(
    tokens["input_ids"],
    skip_special_tokens=False
)

print(decoded)

In [ ]:
import json

def tokenize(example):

    messages = [
        {
            "role": "system",
            "content": """
Read a medical knee MRI report and output exactly 12 labels as JSON.
Use only 0 or 1.

The labels must be exactly:
ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA,
Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.
"""
        },
        {
            "role": "user",
            "content": example["input"]
        },
        {
            "role": "assistant",
            "content": json.dumps(example["output"])
        }
    ]

    # FULL conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # PROMPT only + assistant start
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert to normal Python lists
    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    max_length = 1024

    # Keep prompt + assistant response
    full_ids = full_ids[:max_length]
    prompt_length = min(len(prompt_ids), len(full_ids))

    # Only assistant tokens get loss
    labels = (
        [-100] * prompt_length
        + full_ids[prompt_length:]
    )

    attention_mask = [1] * len(full_ids)

    # Padding
    padding_length = max_length - len(full_ids)

    full_ids += [tokenizer.pad_token_id] * padding_length
    attention_mask += [0] * padding_length
    labels += [-100] * padding_length

    return {
        "input_ids": [int(x) for x in full_ids],
        "attention_mask": [int(x) for x in attention_mask],
        "labels": [int(x) for x in labels]
    }

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(qwen_data)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(qwen_data)

tokenized_dataset = dataset.map(
    tokenize,
    remove_columns=dataset.column_names
)

print(tokenized_dataset.column_names)

In [ ]:
x = tokenized_dataset[0]

label_ids = [
    token for token in x["labels"]
    if token != -100
]

print(tokenizer.decode(
    label_ids,
    skip_special_tokens=False
))

In [ ]:
import torch

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    TrainingArguments,
    Trainer,
    default_data_collator,
)

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen_knee_lora",

    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=10,

    save_strategy="epoch",
    save_total_limit=2,

    report_to="none",

    optim="paged_adamw_8bit",

    remove_unused_columns=False,

    gradient_checkpointing=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=default_data_collator,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("./qwen_knee_lora")
tokenizer.save_pretrained("./qwen_knee_lora")

In [ ]:
import json
import torch

test_report = qwen_data[1]["input"]

messages = [
    {
        "role": "system",
        "content": """
Read a medical knee MRI report and output exactly 12 labels as JSON.
Use only 0 or 1.

The labels must be exactly:
ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA,
Lateral OA, PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture.

Output ONLY the JSON object.
Do not write anything before or after the JSON.
"""
    },
    {
        "role": "user",
        "content": test_report
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(trainer.model.device)

trainer.model.eval()

with torch.no_grad():
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("RAW:")
print(response)

# Submission file

In [ ]:
test = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv")
test_series = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv")
print(test.shape)
study_id = test["StudyInstanceUID"].iloc[0]
test_series[test_series["StudyInstanceUID"] == study_id]

In [ ]:
import pandas as pd
import os

df = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series.csv"
)

train_path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"

row = df.iloc[0]

study_id = row["StudyInstanceUID"]
series_id = row["SeriesInstanceUID"]

series_path = os.path.join(train_path, study_id, series_id)

print("Series:", series_id)
print("Folder exists:", os.path.exists(series_path))
print("DICOM count:", len([f for f in os.listdir(series_path) if f.endswith(".dcm")]))

In [ ]:
import pandas as pd
import os

test = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv"
)

test_series = pd.read_csv(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv"
)

test_path = "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series"

# test.csv-dəki StudyInstanceUID-lərdən birini götür
study_id = test["StudyInstanceUID"].iloc[0]

# həmin StudyInstanceUID-yə aid bütün series-ləri tap
rows = test_series[test_series["StudyInstanceUID"] == study_id]

print("Study:", study_id)
print(rows)

In [ ]:
study_id = test.iloc[0]["StudyInstanceUID"]
print(study_id)

In [ ]:
study_series = test_series[
    test_series["StudyInstanceUID"] == study_id
]

print(study_series)

In [ ]:
study_series

In [ ]:
for _, row in study_series.iterrows():

    series_id = row["SeriesInstanceUID"]

    print("Study:", study_id)
    print("Series:", series_id)

In [ ]:
series_path = os.path.join(
    test_path,
    study_id,
    series_id
)

print(series_path)

In [ ]:
dcm_files = sorted([
    f for f in os.listdir(series_path)
    if f.endswith(".dcm")
])

print("DICOM sayı:", len(dcm_files))
print(dcm_files[:5])

In [ ]:
new_train_file = pd.read_csv("/kaggle/input/datasets/thesharifzade/newtrainfile/new_train_checkpoint.csv")

In [ ]:
# ===================== SUBMISSION: DINOv2 features + KneeClassifier =====================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pydicom
from PIL import Image
from transformers import AutoImageProcessor, AutoModel


# ===================== SETTINGS =====================

COMP = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
DINO_PATH = "/kaggle/input/models/metaresearch/dinov2/pytorch/large/1"

LABELS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_SLICES = 16
BATCH = 32

THRESHOLD = 0.65


# ===================== DINOv2 =====================

processor = AutoImageProcessor.from_pretrained(DINO_PATH)

dino = AutoModel.from_pretrained(DINO_PATH).to(DEVICE)
dino.eval()

if DEVICE == "cuda":
    dino = dino.half()

DTYPE = next(dino.parameters()).dtype


# ===================== LOAD DICOM =====================

def load_slice(path):

    try:
        dcm = pydicom.dcmread(path)

        img = dcm.pixel_array.astype(np.float32)

    except Exception as e:
        return None

    # Əgər image 3 dimensional-dırsa
    if img.ndim == 3:

        if img.shape[-1] in (3, 4):
            img = img.mean(-1)

        else:
            img = img[img.shape[0] // 2]

    # 0-255 normalization
    img = (
        (img - img.min())
        /
        (img.max() - img.min() + 1e-8)
        * 255
    ).astype(np.uint8)

    img = Image.fromarray(img).convert("RGB")

    return img


# ===================== SERIES FEATURE =====================

@torch.no_grad()
def series_feature(series_dir):

    files = sorted(
        f
        for f in os.listdir(series_dir)
        if f.endswith(".dcm")
    )

    if len(files) == 0:
        return None

    # Maximum 16 slice götürürük
    idx = np.linspace(
        0,
        len(files) - 1,
        min(MAX_SLICES, len(files))
    ).astype(int)

    imgs = []

    for i in idx:

        img = load_slice(
            os.path.join(
                series_dir,
                files[i]
            )
        )

        if img is not None:
            imgs.append(img)

    if len(imgs) == 0:
        return None

    feats = []

    for i in range(0, len(imgs), BATCH):

        batch_images = imgs[i:i + BATCH]

        inputs = processor(
            images=batch_images,
            return_tensors="pt"
        )

        pixel_values = inputs["pixel_values"].to(
            DEVICE,
            dtype=DTYPE
        )

        output = dino(
            pixel_values=pixel_values
        )

        # CLS token
        batch_features = output.last_hidden_state[:, 0, :]

        feats.append(
            batch_features.float().cpu()
        )

    # bütün slice feature-lərinin average-i
    feats = torch.cat(feats)

    series_feat = feats.mean(dim=0)

    return series_feat


# ===================== STUDY FEATURE =====================

def study_feature(root, study_id, series_ids):

    feats = []

    for series_id in series_ids:

        series_dir = os.path.join(
            root,
            study_id,
            series_id
        )

        if not os.path.isdir(series_dir):
            continue

        try:

            feature = series_feature(series_dir)

        except Exception as e:

            print(
                "Series error:",
                series_id,
                e
            )

            feature = None

        if feature is not None:
            feats.append(feature)

    if len(feats) == 0:
        return None

    # bütün series feature-lərinin average-i
    study_feat = torch.stack(feats).mean(dim=0)

    return study_feat


# =========================================================
# 1) TRAIN DATA
# =========================================================

train = new_train_file
train_series = pd.read_csv(
    f"{COMP}/train_series.csv"
)


# yalnız bütün 12 label mövcud olan study-lər
labeled = train[
    train[LABELS].notna().all(axis=1)
].reset_index(drop=True)


# Study -> Series list
tr_map = (
    train_series
    .groupby("StudyInstanceUID")["SeriesInstanceUID"]
    .apply(list)
    .to_dict()
)


X = []
y = []


for _, row in labeled.iterrows():

    study_id = row["StudyInstanceUID"]

    series_ids = tr_map.get(
        study_id,
        []
    )

    feature = study_feature(
        f"{COMP}/train_series",
        study_id,
        series_ids
    )

    if feature is None:
        continue

    X.append(feature)

    target = torch.tensor(
        row[LABELS].values.astype("float32")
    )

    y.append(target)


X = torch.stack(X)
y = torch.stack(y)


print(
    "Train X:",
    X.shape
)

print(
    "Train y:",
    y.shape
)


# =========================================================
# 2) CLASSIFIER
# =========================================================

class KneeClassifier(nn.Module):

    def __init__(
        self,
        in_dim=1024,
        n_out=12
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                in_dim,
                512
            ),

            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(
                512,
                n_out
            )
        )

    def forward(self, x):

        return self.network(x)


torch.manual_seed(42)


clf = KneeClassifier(
    in_dim=X.shape[1],
    n_out=len(LABELS)
).to(DEVICE)


criterion = nn.BCEWithLogitsLoss()


optimizer = torch.optim.Adam(
    clf.parameters(),
    lr=1e-4
)


Xd = X.to(DEVICE)

yd = y.to(DEVICE)


# ===================== TRAIN =====================

for epoch in range(200):

    clf.train()

    optimizer.zero_grad()

    logits = clf(Xd)

    loss = criterion(
        logits,
        yd
    )

    loss.backward()

    optimizer.step()


print(
    "Final train loss:",
    round(loss.item(), 4)
)


torch.save(
    clf.state_dict(),
    "/kaggle/working/knee_classifier.pt"
)


# =========================================================
# 3) TEST DATA
# =========================================================

test = pd.read_csv(
    f"{COMP}/test.csv"
)

test_series = pd.read_csv(
    f"{COMP}/test_series.csv"
)


te_map = (
    test_series
    .groupby("StudyInstanceUID")["SeriesInstanceUID"]
    .apply(list)
    .to_dict()
)


# Əgər study oxuna bilməsə istifadə olunacaq prior probability
prior = y.mean(dim=0).numpy()


clf.eval()


rows = []

n_fallback = 0


for study_id in test["StudyInstanceUID"]:

    series_ids = te_map.get(
        study_id,
        []
    )

    feature = study_feature(
        f"{COMP}/test_series",
        study_id,
        series_ids
    )


    # =====================
    # probability
    # =====================

    if feature is None:

        probs = prior.copy()

        n_fallback += 1

    else:

        with torch.no_grad():

            feature = feature.unsqueeze(0).to(DEVICE)

            logits = clf(feature)

            probs = torch.sigmoid(
                logits
            ).cpu().numpy()[0]


    # =====================================================
    # THRESHOLD
    #
    # probability >= 0.65  -> 1
    # probability <  0.65  -> 0
    # =====================================================

    preds = (
        probs >= THRESHOLD
    ).astype(int)


    # submission-a probability YOX,
    # artıq 0 və 1 yazılır
    rows.append(
        [
            study_id,
            *preds
        ]
    )


# =========================================================
# 4) SUBMISSION
# =========================================================

submission = pd.DataFrame(
    rows,
    columns=[
        "StudyInstanceUID",
        *LABELS
    ]
)


submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)


print(
    "Fallback studies:",
    n_fallback,
    "/",
    len(submission)
)

print(
    "Submission shape:",
    submission.shape
)


print("\nPrediction counts:")

for label in LABELS:

    print(
        label,
        "->",
        submission[label].value_counts().to_dict()
    )


submission.head()